# THE FGD AUTO-ENCODER

This implentation is largly taken from genea_numerical_evaluations repository and written by Youngwoo Yoon, based on the original 
FGD proposed by Youngwoo Yoon, Bok Cha, Joo-Haeng Lee, Minsu Jang, Jaeyeon Lee, Jaehong Kim, and Geehyuk Lee in the 2020 paper: 
"Speech Gesture Generation from the Trimodal Context of Text, Audio, and Speaker Identity"

The link to the original, updated the 20/03/2025 is: 
https://github.com/genea-workshop/genea_numerical_evaluations/blob/2022/FGD


This is a collection of the code, with small changes to make it smaller and fit better intro the work we do in this projcet.

### THE 2 PARTS

1. The model design
2. Traing loop

In [14]:
import torch
import torch.nn as nn
import glob
import os

import numpy as np
import torch.nn.functional as F
from torch import optim
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt

In [15]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else 
    "mps" if torch.backends.mps.is_available() else 
    "cpu")

In [16]:
def ConvNormRelu(in_channels, out_channels, downsample=False, padding=0, batchnorm=True):
    if not downsample:
        k = 3
        s = 1
    else:
        k = 4
        s = 2

    conv_block = nn.Conv1d(in_channels, out_channels, kernel_size=k, stride=s, padding=padding)
    norm_block = nn.BatchNorm1d(out_channels)

    if batchnorm:
        net = nn.Sequential(
            conv_block,
            norm_block,
            nn.LeakyReLU(0.2, True)
        )
    else:
        net = nn.Sequential(
            conv_block,
            nn.LeakyReLU(0.2, True)
        )

    return net

In [17]:
class PoseEncoderConv(nn.Module):
    def __init__(self, dim, length):
        super().__init__()

        self.net = nn.Sequential(
            ConvNormRelu(dim, 128, batchnorm=True),
            ConvNormRelu(128, 64, batchnorm=True),
            ConvNormRelu(64, 64, True, batchnorm=True),
            nn.Conv1d(64, 32, 3)
        )

        if length == 30:
            in_channels = 320
        elif length == 60:
            in_channels = 800
        elif length == 90:
            in_channels = 1280
        elif length == 120:
            in_channels = 1760
        elif length == 150:
            in_channels = 2240
        else:
            assert False

        self.out_net = nn.Sequential(
            nn.Linear(in_channels, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(True),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(True),
            nn.Linear(128, 32),
        )

    def forward(self, poses):
        poses = poses.transpose(1, 2)  # to (bs, dim, seq)  # TODO: check if this is correct using the debugger
        out = self.net(poses)
        out = out.flatten(1)
        z = self.out_net(out)

        return z

In [22]:
class PoseDecoderConv(nn.Module):
    def __init__(self, dim, length):
        super().__init__()

        if length == 30:
            out_channels = 120
        elif length == 60:
            out_channels = 240
        elif length == 90:
            out_channels = 360
        elif length == 120:
            out_channels = 480
        elif length == 150:
            out_channels = 600
        else:
            assert False

        self.pre_net = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(True),
            nn.Linear(64, out_channels),
        )

        self.net = nn.Sequential(
            nn.ConvTranspose1d(4, 32, 3),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2, True),
            nn.ConvTranspose1d(32, 32, 3),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2, True),
            nn.Conv1d(32, 32, 3),
            nn.Conv1d(32, dim, 3),
        )

    def forward(self, feat):
        out = self.pre_net(feat)
        out = out.view(feat.shape[0], 4, -1)
        out = self.net(out)
        out = out.transpose(1, 2)
        return out

In [19]:
class EmbeddingNet(nn.Module):
    def __init__(self, pose_dim, n_frames):
        super().__init__()
        self.pose_encoder = PoseEncoderConv(pose_dim, n_frames)
        self.decoder = PoseDecoderConv(pose_dim, n_frames)

    def forward(self, poses):
        poses_feat = self.pose_encoder(poses)
        out_poses = self.decoder(poses_feat)
        return poses_feat, out_poses

### TRIAINING LOOP

This is NOT taken from the genea_numerical_evaluations repository, but lossfunctions, optimaser and learningrate is by default.

In [ ]:
from tqdm import tqdm
from IPython.display import clear_output
import time
from collections import defaultdict

# This is mostly created by combining elements from Solved_MLP_mnist_tensorflow-pytorch.ipynb 
# and SOLUTION_Convolutional_networks.ipynb from the applied AI course.

def train(
        model_embedding_net_AE: EmbeddingNet,
        training_loader,
        val_loader, 
        num_epochs: int, 
        lr=0.0003,
        loss_f=nn.HuberLoss(),
):

    nn.L1Loss()
    # Add profiling data structures
    profiling = defaultdict(list)
    visalize_step = 50  # How often to print profiling stats

    optimizer = torch.optim.AdamW(model_embedding_net_AE.parameters(), lr=lr)

    torch.set_float32_matmul_precision('high')
    
    # The current lowest validation loss gets defined as an infinitely large number in order to make sure that 
    # it gets reduced in the first epoch. .
    current_min_val_loss = np.inf

    # I then move the model to the device that is being used and put in traning mode
    model_embedding_net_AE = model_embedding_net_AE.to(device)
    model_embedding_net_AE = torch.compile(model_embedding_net_AE, backend="cudagraphs")
    print("Model compling to cudagraphs runtime")
    model_embedding_net_AE.train()
    
    # I then define a map of lists used for tracking the training and validation loss for each epoch. 
    # I'll later use these two sets to plot the progress of the model training.
    loss_rec = {'train' : [], 'val' : [], 'train_plot': []}
    
    # This is the main training loop that goes through the entire dataset and trains the model on it for each epoch.
    for epoch in range(num_epochs):
        progress_bar = tqdm(training_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=True)

        # I reset the training loss for each epoch.
        epoch_train_loss = 0
        
        # Start batch timer
        batch_start_time = time.time()

        # During the epoch, all the data items are iterated over.
        for i, batch_data in enumerate(progress_bar):
            # Data loading time
            data_load_time = time.time() - batch_start_time
            profiling["data_loading"].append(data_load_time)
            

            reshape_time = time.time()

            # IMPORTANT CHANGE: Handle pre-batched data from RAMResidentDataset
            # Each item contains a full batch already, so we just need to unpack the tuple
            # and handle the extra dimension from batch_size=1
            gesture_sequence, seed_gesture, audio_features, main_agent_id_one_hot = [
                item.squeeze(0).to(device) for item in batch_data
            ]

            reshape_time = time.time() - reshape_time
            profiling["data_reshaping"].append(reshape_time)

            # Model forward time
            forward_start = time.time()
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                output = model_embedding_net_AE(
                    poses = gesture_sequence
                )
            forward_time = time.time() - forward_start
            profiling["model_forward"].append(forward_time)

            # Loss calculation time
            loss_start = time.time()

            l1_loss = nn.L1Loss(reduction='none')
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                recon_loss = torch.mean(l1_loss(output, gesture_sequence), dim=(1, 2))  # calc mean over spatial dims

                # Add temporal diff loss - aka ponish of the making wrong changes between frames
                target_diff = gesture_sequence[:, 1:] - gesture_sequence[:, :-1]
                recon_diff = output[:, 1:] - output[:, :-1]
                recon_loss += torch.mean(l1_loss(recon_diff, target_diff), dim=(1, 2))  

                recon_loss = torch.sum(recon_loss)  # Sum over all elements

            loss_time = time.time() - loss_start
            profiling["loss_calculation"].append(loss_time)

            epoch_train_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})
            loss_rec['train'].append(loss.item())

            if i % visalize_step == 0:
                
                clear_output(wait=True)
                visualisation_start = time.time()

                # add the averaged loss over hte last visalize_step to the loss_rec['train_plot']
                loss_rec['train_plot'].append(np.mean(loss_rec['train'][-visalize_step:]))
                
                # Visualization code remains unchanged
                fig, axs = plt.subplots(1, 5, figsize=(30, 6))

                cmap = 'viridis'
                vmin = -1
                vmax = 1

                axs[0].imshow(output.to(torch.float32).permute(0, 2, 1)[0, :, :].cpu().detach().numpy(), cmap=cmap, vmin=vmin, vmax=vmax)
                axs[0].set_title("Output tensor")
                axs[0].text(10, 190, f"Max: {torch.max(output):.4f}", color="black")
                axs[0].text(10, 200, f"Min: {torch.min(output):.4f}", color="black")
                axs[0].text(10, 210, f"Mean: {torch.mean(output):.4f}", color="black")

                axs[1].imshow(gesture_sequence.to(torch.float32).permute(0, 2, 1)[0, :, :].cpu().detach().numpy(), cmap=cmap, vmin=vmin, vmax=vmax)
                axs[1].set_title("Actual gesture")
                axs[1].text(10, 190, f"Max: {torch.max(gesture_sequence):.4f}", color="black")
                axs[1].text(10, 200, f"Min: {torch.min(gesture_sequence):.4f}", color="black")
                axs[1].text(10, 210, f"Mean: {torch.mean(gesture_sequence):.4f}", color="black")

                axs[2].imshow((gesture_sequence - output).to(torch.float32).permute(0, 2, 1)[0, :, :].cpu().detach().numpy(), cmap=cmap, vmin=vmin, vmax=vmax)
                axs[2].set_title("Difference between output and actual gesture")
                axs[2].text(10, 190, f"Max: {torch.max(gesture_sequence - output):.4f}", color="black")

                axs[3].plot(loss_rec['train_plot'])
                axs[3].set_title('Training Loss')
                axs[3].set_xlabel('Epoch')
                axs[3].set_ylabel('Loss')
                axs[3].set_yscale('log')
                axs[3].grid(True)
                
                # Add text with profiling data to the loss plot
                avg_data_time = np.mean(profiling["data_loading"][-visalize_step:]) * 1000
                avg_reshape_time = np.mean(profiling["data_reshaping"][-visalize_step:]) * 1000
                avg_device_time = np.mean(profiling["device_transfer"][-visalize_step:]) * 1000
                avg_diffusion_time = np.mean(profiling["diffusion_forward"][-visalize_step:]) * 1000
                avg_forward_time = np.mean(profiling["model_forward"][-visalize_step:]) * 1000
                avg_loss_time = np.mean(profiling["loss_calculation"][-visalize_step:]) * 1000
                avg_backward_time = np.mean(profiling["backward"][-visalize_step:]) if profiling["backward"] else 0
                avg_optimizer_time = np.mean(profiling["optimizer"][-visalize_step:]) if profiling["optimizer"] else 0
                
                profiling_text = (
                    f"PROFILING (ms/batch):\n"
                    f"Data loading: {avg_data_time:.2f}\n"
                    f"Data reshaping: {avg_reshape_time:.2f}\n"
                    f"Device transfer: {avg_device_time:.2f}\n"
                    f"Diffusion forward: {avg_diffusion_time:.2f}\n"
                    f"Model forward: {avg_forward_time:.2f}\n"
                    f"Loss calculation: {avg_loss_time:.2f}\n"
                    f"Backward: {avg_backward_time:.2f}\n"
                    f"Optimizer: {avg_optimizer_time:.2f}"
                )
                axs[4].text(0.02, 0.98, profiling_text, transform=axs[4].transAxes, 
                            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
                plt.show()

                visualisation_time = time.time() - visualisation_start
                print(f"Visualisation time: {visualisation_time:.2f} s")

            # Backward pass time
            backward_start = time.time()
            optimizer.zero_grad()
            loss.backward()
            backward_time = time.time() - backward_start
            profiling["backward"].append(backward_time)

            # Optimizer step time
            optimizer_start = time.time()
            optimizer.step()
            optimizer_time = time.time() - optimizer_start
            profiling["optimizer"].append(optimizer_time)
            
            # Start timing for next batch
            batch_start_time = time.time()

        
        # After the training loop, the launch on the validation set is calculated in the same way as on 
        # the training set, but without the gradient descent and transformation of the model premises.
        
        epoch_val_loss = 0
        # model.eval()
        # for images_v, labels_v in loaders['validation']:
        #     if torch.cuda.is_available():
        #        images_v, labels_v = images_v.cuda(), labels_v.cuda()
        #     output = model(images_v)
        #     loss_v = loss_f(output, labels_v)
        #     epoch_val_loss += loss_v.item()
        
        # I then take the average of the training loss and validation loss.
        train_loss = epoch_train_loss / len(training_loader)
        # val_loss = epoch_val_loss / len(loaders["validation"])
        
        # And format them using print statements and output them after the epoch is finished.
        # print(f'Epoch {epoch+1}')
        # print(f'Training Loss: {train_loss}')
        # print(f'Validation Loss: ???') # {val_loss}')
        # print('-------------------')
        
        # Finally, I record the loss and append them to the list used for plotting the loss later.
        # loss_rec['train'].append(train_loss)
        # loss_rec['val'].append(val_loss)
        
        # If the validation loss is smaller than the previously best validation loss, the model is saved to a 
        # separate file in the same folder as this, or given in the save function. 
        # if train_loss < current_min_val_loss:
        #     print(f'train Loss Decreased({current_min_val_loss}--->{train_loss}) \t Saving The Model')
        #     current_min_val_loss = train_loss
        #     # Saving State Dict
        #     torch.save(model.state_dict(), 'saved_model.pth')

        # draw the training loss
        # print(len(loss_rec['train']))
        # clear_output(wait=True)
    
    # When all of the epochs are over, the entire list of training loss and validation loss are returned.
    return loss_rec['train'] #, loss_rec['val']

### Using the traing loop and model

In [24]:
from dataset.dataset import *

model_AE = EmbeddingNet(pose_dim=32, n_frames=150)  # pose_features_per_frame and n_gesture_length of our model


train_loss = train(
    model                           = model_AE,
    training_loader                 = DataLoader(
                                            RAMResidentDataset(
                                                    folder = "dataset/genea2023_dataset/trn/main-agent/features",
                                                    windows_file="dataset/genea2023_dataset/trn/main-agent/training_windows_100k.npz",
                                                    batch_size=128,  
                                                    epoch_length=1000
                                            ),
                                            batch_size = 1,
                                            num_workers = 0,
                                            pin_memory = True
                                    ),
    val_loader                      = None, 
    num_epochs                      = 1000,
    lr                              = 0.00008,
    loss_f                          = nn.HuberLoss(),
)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/genea2023_dataset/trn/main-agent/training_windows_100k_meta.pkl'